<a href="https://colab.research.google.com/github/nilotpal-makes-stuff/TensorFlow-2-study/blob/main/youtube-ML_Zero_to_Hero/04Build_an_image_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML Zero to Hero
Youtube - https://www.youtube.com/watch?v=KNAWp2S3w94&list=PLQY2H8rRoyvwWuPiWnuTDBHe7I0fMSsfO
## Build an Image Classifier
In this chapter we will build a Neural Network model which will take an image of a person's hand and identify whether the person is forming their hand in 'Rock', 'Paper' or 'Scissors' shape.

We download the dataset using the following code :

In [1]:
!wget --no-check-certificate \
    https://storage.googleapis.com/learning-datasets/rps.zip \
    -O /tmp/rps.zip

!wget --no-check-certificate \
    https://storage.googleapis.com/learning-datasets/rps-test-set.zip \
    -O /tmp/rps-test-set.zip

--2026-04-02 17:37:17--  https://storage.googleapis.com/learning-datasets/rps.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 64.233.188.207, 64.233.189.207, 108.177.97.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|64.233.188.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 200682221 (191M) [application/zip]
Saving to: ‘/tmp/rps.zip’

/tmp/rps.zip        100%[===================>] 191.38M  19.7MB/s    in 11s     

2026-04-02 17:37:29 (17.0 MB/s) - ‘/tmp/rps.zip’ saved [200682221/200682221]

--2026-04-02 17:37:29--  https://storage.googleapis.com/learning-datasets/rps-test-set.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 64.233.188.207, 64.233.189.207, 108.177.97.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|64.233.188.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 29516758 (28M) [application/zip]
Saving to: ‘/tmp/rps-test-set.zip’

/tmp/rps-

Then we extract the contents of the zip file using :

In [2]:
"""
Unzipping files in the downloaded zip files into '/tmp' directory
Training data = '/tmp/rps'
Testing data = '/tmp/rps-test-set'
"""

import os
import zipfile

#Training data
local_zip = '/tmp/rps.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('/tmp/')
zip_ref.close()

#Testing data
local_zip = '/tmp/rps-test-set.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('/tmp/')
zip_ref.close()


Now, the zip file has the following structure :

Training Data
<pre>
  rps
    - paper
      - paper[01-09]-[001-999].png
    - rock
      - rock[01-09]-[001-999].png
    - scissors
      - scissors[01-09]-[001-999].png
</pre>
Testing data
<pre>
  rps-test-set
    - paper
      - testpaper[01-09]-[001-999].png
    - rock
      - testrock[01-09]-[001-999].png
    - scissors
      - testscissors[01-09]-[001-999].png
</pre>

In [3]:
"""
Create lists of all image files for each category from Training dataset
"""
rock_dir = os.path.join('/tmp/rps/rock')
paper_dir = os.path.join('/tmp/rps/paper')
scissors_dir = os.path.join('/tmp/rps/scissors')

print('total training rock images:', len(os.listdir(rock_dir)))
print('total training paper images:', len(os.listdir(paper_dir)))
print('total training scissors images:', len(os.listdir(scissors_dir)))

rock_files = os.listdir(rock_dir)
rock_files = [os.path.join(rock_dir, f) for f in rock_files]
print(rock_files[:10])

paper_files = os.listdir(paper_dir)
paper_files = [os.path.join(paper_dir, f) for f in paper_files]
print(paper_files[:10])

scissors_files = os.listdir(scissors_dir)
scissors_files = [os.path.join(scissors_dir, f) for f in scissors_files]
print(scissors_files[:10])

total training rock images: 840
total training paper images: 840
total training scissors images: 840
['/tmp/rps/rock/rock05ck01-066.png', '/tmp/rps/rock/rock06ck02-068.png', '/tmp/rps/rock/rock04-010.png', '/tmp/rps/rock/rock01-050.png', '/tmp/rps/rock/rock03-060.png', '/tmp/rps/rock/rock01-113.png', '/tmp/rps/rock/rock02-006.png', '/tmp/rps/rock/rock04-075.png', '/tmp/rps/rock/rock04-005.png', '/tmp/rps/rock/rock02-010.png']
['/tmp/rps/paper/paper04-047.png', '/tmp/rps/paper/paper05-051.png', '/tmp/rps/paper/paper04-042.png', '/tmp/rps/paper/paper07-000.png', '/tmp/rps/paper/paper02-039.png', '/tmp/rps/paper/paper05-103.png', '/tmp/rps/paper/paper03-072.png', '/tmp/rps/paper/paper01-055.png', '/tmp/rps/paper/paper03-030.png', '/tmp/rps/paper/paper02-062.png']
['/tmp/rps/scissors/testscissors01-094.png', '/tmp/rps/scissors/testscissors03-014.png', '/tmp/rps/scissors/testscissors01-079.png', '/tmp/rps/scissors/scissors03-077.png', '/tmp/rps/scissors/testscissors01-110.png', '/tmp/rps/sc

Now we use the **Image** module from the **Pillow library (PIL)** to read the images and then convert them into numpy arrays. Each image is 300x300 in size with 4 color channels - Red Green Blue Alpha (RGBA). There are a total of 840 images for each category. Hence the shape of the numpy array will be (840,300,300,4) - (num_images, height, width, num_channels).

Since we donot need RGBA color information in the image to determine the shape that the hand is trying to depict, we convert each image to grayscale. We can get the grayscale image using the **image.convert('L')** method. Without the color channels, the numpy array will have the shape (840, 300, 300).

In [4]:
"""
Use Pillow library to convert image files to numpy arrays
"""
from PIL import Image
import numpy

rock_images = numpy.array([numpy.array(Image.open(imgfile).convert('L')) for imgfile in rock_files])
paper_images = numpy.array([numpy.array(Image.open(imgfile).convert('L')) for imgfile in paper_files])
scissors_images = numpy.array([numpy.array(Image.open(imgfile).convert('L')) for imgfile in scissors_files])

#array shape - (num_images, shape_img[0], shape_img[1], num_channels)
print(rock_images.shape)
print(paper_images.shape)
print(scissors_images.shape)


(840, 300, 300)
(840, 300, 300)
(840, 300, 300)


In [5]:
"""
View image arrays attributes for validation
"""
import random

num_images = 840
index = random.randint(0,num_images)

#Rock images
#Shape should be (300,300)
print(rock_images[index].shape)
#Min Max value should be in range [0,255]
print(max(rock_images[index].flatten()), min(rock_images[index].flatten()))
print()

#Paper images
#Shape should be (300,300)
print(paper_images[index].shape)
#Min Max value should be in range [0,255]
print(max(paper_images[index].flatten()), min(paper_images[index].flatten()))
print()

#Scissors images
#Shape should be (300,300)
print(scissors_images[index].shape)
#Min Max value should be in range [0,255]
print(max(scissors_images[index].flatten()), min(scissors_images[index].flatten()))
print()

#Display image array
print(rock_images[index])
print()
print(paper_images[index])
print()
print(scissors_images[index])
print()

(300, 300)
255 14

(300, 300)
255 95

(300, 300)
255 24

[[255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 254]
 ...
 [255 255 255 ... 254 254 254]
 [255 255 255 ... 254 254 253]
 [255 255 255 ... 254 254 253]]

[[255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 ...
 [255 255 255 ... 254 254 254]
 [255 255 255 ... 254 254 254]
 [255 255 255 ... 254 254 253]]

[[254 253 255 ... 251 250 250]
 [253 255 253 ... 252 251 250]
 [254 253 254 ... 252 251 252]
 ...
 [253 252 252 ... 247 248 249]
 [253 254 251 ... 248 248 248]
 [252 253 252 ... 248 247 249]]



Now we normalize the image arrays so that the Neural Network model can provide us more accurate results.

In [6]:
"""
Normalise image arrays
"""
rock_images_norm = rock_images/255.0
paper_images_norm = paper_images/255.0
scissors_images_norm = scissors_images/255.0

Now we construct our Convolution Neural Network Model. This time we use 4 stacks of Conv2D and MaxPooling2D layers as the image is larger (300x300) - This will give us more accurate results. We also use a **Dropout(0.5)** layer to remove some of the neurons in the CNN - this helps improve efficiency.

The **tensorflow.keras.layers.Dropout** layer randomly sets input units to 0 with a frequency of rate at each step during training time to prevent overfitting, while scaling remaining inputs by 1 / (1 - rate) to maintain the expected sum. This layer is inactive during inference (testing/prediction), where no values are dropped.

Syntax : tensorflow.keras.layers.Dropout(rate, noise_shape=None, seed=None)
* **rate** - float [0,1] - fraction of units to drop
* **noise_shape** - 1d int tensor - shape of binary dropout mask, useful for sharing noise across specific dimensions
* **seed** - int - random seed for reproducibility

In [7]:
import tensorflow

model = tensorflow.keras.models.Sequential([
    #Convolution network
    tensorflow.keras.layers.Conv2D(64, (3,3), activation='relu', input_shape=(300, 300, 1)),
    tensorflow.keras.layers.MaxPooling2D(2, 2),
    tensorflow.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tensorflow.keras.layers.MaxPooling2D(2, 2),
    tensorflow.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tensorflow.keras.layers.MaxPooling2D(2, 2),
    tensorflow.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tensorflow.keras.layers.MaxPooling2D(2, 2),

    #deep neural network
    tensorflow.keras.layers.Flatten(),
    tensorflow.keras.layers.Dropout(0.5),
    tensorflow.keras.layers.Dense(512, activation='relu'),
    tensorflow.keras.layers.Dense(3, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Compile the model :

In [8]:
#model compile
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

Next we fit the model using the training data.

In [10]:
#Consolidate data
images_test_norm = numpy.concatenate((rock_images_norm, paper_images_norm, scissors_images_norm))
#print(images_norm.shape)
labels = numpy.array([0 for _ in rock_images] + [1 for _ in paper_images] + [2 for _ in scissors_images], dtype=numpy.uint8)
#labels = tensorflow.keras.utils.to_categorical(labels, 3)
print(labels.shape)
print(images_test_norm.shape)
#Fit model
model.fit(images_test_norm, labels, epochs=5)

(2520,)
(2520, 300, 300)
Epoch 1/5


ValueError: Creating variables on a non-first call to a function decorated with tf.function.